# Which Algorithm Wins?

In [01_first_model.ipynb](01_first_model.ipynb) we built one model with defaults. Now we'll try every algorithm, pick the best one, and tune it.

**What you'll learn:**
- Regression (predicting a number, not a category)
- Benchmark all algorithms at once with `screen()`
- Optimize hyperparameters with `tune()`
- Compare models fairly with `compare()`

**New verbs:** `screen`, `tune`, `compare`

In [1]:
import ml

## 1. The Dataset

California Housing — 20,640 homes with 8 features. The target is `price` (median house value in $100k). This is **regression** — predicting a continuous number instead of a category.

In [2]:
data = ml.dataset("houses")
data.head(3)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521


In [3]:
ml.profile(data, "price")

── Profile [regression] ────────────────
Rows:     20,640
Columns:  9
Target:   price

  ! High condition number (9415). Near-collinear features.



**regression** — auto-detected because `price` is numeric. The "high condition number" warning means some features span very different scales (e.g., population in the thousands vs latitude around 35). Tree-based algorithms handle this fine; for linear models, the library auto-scales features.

## 2. Baseline — Start with Defaults

Same workflow as Notebook 1: split, fit, evaluate.

In [4]:
s = ml.split(data, "price", seed=42)
model = ml.fit(s.train, "price", seed=42)
print(model)

── Model [xgboost · regression] ────────
Target:   price
Features: 8
Rows:     12,384  (0.08s)
Seed:     42
Hash:     b5214c51



In [5]:
ml.evaluate(model, s.valid)

── Metrics [regression] ────────────────
rmse:  0.4719
mae:   0.3157
r2:    0.8294
(0.01s)

Regression uses different metrics than classification:
- **RMSE** (0.47) — average prediction error, in the same units as the target ($47k off, since prices are in $100k)
- **MAE** (0.32) — same idea, but less sensitive to big misses ($32k average error)
- **R2** (0.83) — how much of the variation the model explains. 1.0 = perfect, 0.0 = useless. 0.83 is good.

But is XGBoost the best choice? Maybe a simpler algorithm works just as well — or maybe something else does better.

## 3. Screen — Try Everything

`screen()` trains every available algorithm with default settings and ranks them. One line, one leaderboard.

In [6]:
leaderboard = ml.screen(s, "price")
leaderboard

── Screen [regression · 8 algorithms] ──
    algorithm   rmse    mae     r2  time
     catboost 0.4550 0.3024 0.8414  0.90
      xgboost 0.4719 0.3157 0.8294  0.09
     lightgbm 0.4722 0.3168 0.8292  0.08
random_forest 0.5157 0.3409 0.7963  4.46
          svm 0.6085 0.4060 0.7163  2.44
          knn 0.6705 0.4549 0.6556  0.06
       linear 0.7298 0.5336 0.5920  0.03
  elastic_net 1.0227 0.8065 0.1988  0.02

You may see scaling warnings for SVM, KNN, or Elastic Net — this is normal. The library automatically standardizes features for algorithms that need it, and tells you it's happening.

The leaderboard tells a clear story:

1. **Tree-based models dominate** — CatBoost, XGBoost, LightGBM, Random Forest all use decision trees internally. They handle non-linear relationships and feature interactions naturally.

2. **CatBoost leads** (R2=0.84) — but only slightly better than XGBoost (R2=0.83). The difference is small with default settings.

3. **Linear models struggle** — Elastic Net (R2=0.20) is nearly useless because house prices have complex non-linear patterns that a straight line can't capture.

4. **Speed varies wildly** — Random Forest takes 4.5s vs LightGBM's 0.08s for similar performance. In production, that matters.

The leaderboard is a full pandas DataFrame — you can filter it (`leaderboard[leaderboard['r2'] > 0.8]`), sort it, or export it with `.to_csv()`.

The default choice (XGBoost) is a solid pick. But can we make it *better*?

## 4. Tune — Optimize the Winner

Every algorithm has settings ("hyperparameters") that affect performance: how deep the trees grow, how fast it learns, how much data each tree sees. Defaults are reasonable, but tuning these settings can squeeze out more performance.

`tune()` tries 20 random combinations and picks the best one using cross-validation (training multiple models on different subsets of the data to get a more reliable estimate).

In [7]:
tuned = ml.tune(s.train, "price", algorithm="xgboost", seed=42, n_trials=20)
print(tuned)

── Tuned [xgboost · regression] ────────
Target:   price      max_depth: 9               colsample_bytree: 0.8365
Features: 8          learning_rate: 0.1002      min_child_weight: 6
Trials:   20         n_estimators: 265          reg_alpha: 8.8721
Best CV:  0.4692     subsample: 0.8664          reg_lambda: 4.7221
Seed:     42
Hash:     e056df27



The tuner found settings that give a Best CV RMSE of 0.4692 (down from 0.4719 with defaults). Let's verify on the validation set.

In [8]:
ml.evaluate(tuned, s.valid)

── Metrics [regression] ────────────────
rmse:  0.4503
mae:   0.2970
r2:    0.8447
(0.03s)

Tuning improved every metric:
- RMSE: 0.4719 &rarr; 0.4503 (predictions are $2k closer on average)
- R2: 0.8294 &rarr; 0.8447 (explains 1.5% more variation)

Not dramatic, but free performance from 20 trials of automated search.

## 5. Compare — Fair Side-by-Side

`compare()` puts multiple models next to each other, evaluated on the same data. No re-fitting — just evaluation.

In [9]:
ml.compare([model, tuned], s.valid)

── Compare [regression · 2 models] ─────
 # algorithm   rmse    mae     r2  time
 2   xgboost 0.4503 0.2970 0.8447  0.03
 1   xgboost 0.4719 0.3157 0.8294  0.01

The `#` column shows the model number (in the order you passed them). The tuned model (#2) wins across all metrics. Both are XGBoost — same algorithm, different hyperparameters.

## 6. Explain — What Drives House Prices?

Let's peek inside the winning model.

In [10]:
ml.explain(model)

── Explain [xgboost] ───────────────────
  MedInc      ████████████████████   46.6%
  AveOccup    ██████▌                15.3%
  Longitude   ████▌                  11.4%
  Latitude    ████                   10.0%
  HouseAge    ██▌                     6.7%
  AveRooms    █▌                      4.4%
  AveBedrms   █                       3.2%
  Population  █                       2.4%

**Median income is king** (47%) — wealthy neighborhoods have expensive houses. Location (Latitude + Longitude = 21%) is second. This matches common sense: in real estate, it's income and location.

Population barely matters (2.4%) — the number of people nearby doesn't predict price well.

## 7. Predict — What Does the Model Say?

Let's use the tuned model to actually predict some house prices.

In [ ]:
preds = ml.predict(tuned, s.valid)
comparison = s.valid[["price"]].copy()
comparison["predicted"] = preds.values
comparison["error"] = (comparison["predicted"] - comparison["price"]).abs()
comparison.head(8)

---

## Recap

The exploration loop:

```python
leaderboard = ml.screen(s, "price")                                    # try everything
winner = leaderboard.iloc[0]["algorithm"]                              # pick the best
tuned  = ml.tune(s.train, "price", algorithm=winner, seed=42)         # optimize it
ml.compare([model, tuned], s.valid)                                    # compare fairly
preds  = ml.predict(tuned, s.valid)                                    # use it
```

`screen()` finds the best algorithm. `tune()` optimizes its settings. `compare()` proves the improvement. `predict()` puts it to work.

**What's next?** Real data has missing values, categories, and class imbalance. In [03_real_world.ipynb](03_real_world.ipynb), we'll handle messy data and learn about the evaluate/assess distinction — the most important concept in honest ML evaluation.